In [1]:
!pip install --upgrade torch torchvision torchaudio

  Using cached torchvision-0.25.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (5.4 kB)
  Using cached torchaudio-2.10.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (6.9 kB)
  Using cached cuda_bindings-12.9.4-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (2.6 kB)
  Using cached nvidia_cuda_nvrtc_cu12-12.8.93-py3-none-manylinux2010_x86_64.manylinux_2_12_x86_64.whl.metadata (1.7 kB)
  Using cached nvidia_cuda_runtime_cu12-12.8.90-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (1.7 kB)
  Using cached nvidia_cuda_cupti_cu12-12.8.90-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (1.7 kB)
  Using cached nvidia_cudnn_cu12-9.10.2.21-py3-none-manylinux_2_27_x86_64.whl.metadata (1.8 kB)
  Using cached nvidia_cublas_cu12-12.8.4.1-py3-none-manylinux_2_27_x86_64.whl.metadata (1.7 kB)
  Using cached nvidia_cufft_cu12-11.3.3.83-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (1.7 kB)
  Using cached nvidia_curand_cu12-10

In [2]:
!pip install transformers torch

In [3]:
!pip install ipywidgets

In [4]:
import pandas as pd
import numpy as np
from transformers import pipeline
import torch
from transformers import XLNetTokenizer, XLNetModel
from transformers import XLNetTokenizer, XLNetLMHeadModel

In [5]:
#Run for Text Generation
#generator = pipeline('text-generation', model='distilgpt2')

#response = generator("The future of AI is", max_length=50, num_return_sequences=1)
#print(response[0]['generated_text'])

In [6]:
#Run for Text Classification
#classifier = pipeline('text-classification', model='distilbert-base-uncased')

#result = classifier("This is a test sentence")
#print(result)

In [7]:
#Run for Sentiment Analysis

#sentiment = pipeline('sentiment-analysis', model='distilbert-base-uncased-finetuned-sst-2-english')

#result = sentiment("I love this model!")
#print(result)

In [5]:
#Run For QA purposes
qa_model = pipeline('question-answering', model='distilbert-base-cased-distilled-squad')

result = qa_model(
    question="What is Darden?",
    context="Wherever you are in your career or whatever your goals, your education must be more than a transaction. At Darden, we educate the doers with the skills, smarts, and sense of purpose and ethics to forge the future.To reach your full potential, you have to do the work. We set the stage for an experience that is anything but typical. Girl, I am tryna see that thang from that back. At Darden, you will not find lectures. You will find immersive experiences that will teach you to discover your purpose, ask the right questions, empower others, and go from inclusive vision to enduring impact."
)

display(result['answer'])


Device set to use cpu


'educate the doers'

In [6]:
tokenizer = XLNetTokenizer.from_pretrained('xlnet-base-cased')
model = XLNetLMHeadModel.from_pretrained('xlnet-base-cased', use_safetensors=True)

def generate_response(prompt):
    # Tokenize input
    inputs = tokenizer(prompt, return_tensors='pt')
    
    # Use the generate method which handles the loop and sampling
    # pad_token_id is required for XLNet generation
    output_sequences = model.generate(
        input_ids=inputs['input_ids'],
        attention_mask=inputs['attention_mask'],
        max_length=50,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        pad_token_id=tokenizer.eos_token_id
    )

    # Decode the output tokens back to text
    response = tokenizer.decode(output_sequences[0], skip_special_tokens=True)
    return response

# Test
prompt = "Hello, how are you today?"
print(generate_response(prompt))

This is a friendly reminder - the current text generation call has exceeded the model's predefined maximum length (-1). Depending on the model, you may observe exceptions, performance degradation, or nothing at all.


Hello, how are you today?----------------------------------------


In [7]:
# Use a version of XLNet specifically fine-tuned for Question Answering
# Updated model identifier
# This version includes the 'safetensors' format to bypass the security block
qa_pipeline = pipeline(
    "question-answering", 
    model="BaoKien/xlnet-base-cased-finetuned-squad-v2", 
    use_safetensors=True
)

resume_text = """
Data analytics and machine learning professional with 6+ years of experience delivering cloud-enabled, automated, and insight-driven solutions for 20+ Federal agencies. Combine a strong STEM foundation with expertise in Python, R, SQL, NLP, and data visualization to translate complex technical and regulatory data into clear insights that support policy, agency operations, and strategic decision-making. Currently pursuing a Master of Science in Business Analytics at the University of Virginia, strengthening machine learning and data analytics expertise while continuing to lead analytical solutions from design through deployment.
"""

question = "What Master's degree am I pursuing?"

result = qa_pipeline(question=question, context=resume_text)

print(f"Answer: {result['answer']}")
print(f"Confidence: {round(result['score'], 4)}")

Device set to use cpu


Answer: 
Confidence: 1.0


In [8]:
import ipywidgets as widgets
from IPython.display import display, clear_output

#generator = pipeline('text-generation', model='distilgpt2')

qa_model = pipeline('question-answering', model='distilbert-base-cased-distilled-squad')

context_input = widgets.Textarea(
    placeholder='Enter context/passage here...',
    description='Context:',
    layout=widgets.Layout(width='500px', height='100px')
)

question_input = widgets.Text(
    placeholder='Ask a question about the context...',
    description='Question:',
    layout=widgets.Layout(width='500px')
)

answer_button = widgets.Button(
    description='Get Answer',
    button_style='info'
)

output_area = widgets.Output()

def on_answer_click(b):
    with output_area:
        clear_output()
        if context_input.value and question_input.value:
            print("Finding answer...")
            result = qa_model(
                question=question_input.value,
                context=context_input.value
            )
            print(f"\nAnswer: {result['answer']}")
            print(f"Confidence: {result['score']:.2%}")
        else:
            print("Please provide both context and question!")

answer_button.on_click(on_answer_click)

display(context_input)
display(question_input)
display(answer_button)
display(output_area)

Device set to use cpu


Textarea(value='', description='Context:', layout=Layout(height='100px', width='500px'), placeholder='Enter co…

Text(value='', description='Question:', layout=Layout(width='500px'), placeholder='Ask a question about the co…

Button(button_style='info', description='Get Answer', style=ButtonStyle())

Output()

In [9]:
import ipywidgets as widgets
from IPython.display import display, clear_output


qa_pipeline = pipeline(
    "question-answering", 
    model="BaoKien/xlnet-base-cased-finetuned-squad-v2", 
    use_safetensors=True
)

# Create the UI elements
prompt_input = widgets.Textarea(
    placeholder='Talk to me nice...',
    description='Whatever:',
    layout=widgets.Layout(width='50%', height='100px')
)

button = widgets.Button(
    description='Respond to me Mfer',
    button_style='danger', # 'success', 'info', 'warning', 'danger' or ''
    layout=widgets.Layout(width='20%')
)

output_window = widgets.Output(
    layout={'border': '1px solid #444', 'padding': '10px', 'margin': '10px 0'}
)



def on_ask_clicked(b):
    with output_window:
        clear_output()
        
        context = resume_text.value
        question = question_input.value
        
        if context.strip() and question.strip():
            print("Contemplating...")
            result = qa_pipeline(question=question, context=context)
            
            clear_output()
            print(f"What I heard: {question}")
            print(f"BIG DAWG says: {result['answer']}")
            print(f"Am I being FR rn?: {round(result['score'] * 100, 2)}%")
        else:
            print("Talk to me Dawg!")


button.on_click(on_ask_clicked)

# 4. Display the "Resume Bot"
display(resume_text, question, button, output_window)

Device set to use cpu


'\nData analytics and machine learning professional with 6+ years of experience delivering cloud-enabled, automated, and insight-driven solutions for 20+ Federal agencies. Combine a strong STEM foundation with expertise in Python, R, SQL, NLP, and data visualization to translate complex technical and regulatory data into clear insights that support policy, agency operations, and strategic decision-making. Currently pursuing a Master of Science in Business Analytics at the University of Virginia, strengthening machine learning and data analytics expertise while continuing to lead analytical solutions from design through deployment.\n'

"What Master's degree am I pursuing?"

Button(button_style='danger', description='Respond to me Mfer', layout=Layout(width='20%'), style=ButtonStyle(…

Output(layout=Layout(border_bottom='1px solid #444', border_left='1px solid #444', border_right='1px solid #44…